In [1]:
import pandas as pd
import numpy as np
import os
import re
from collections import defaultdict

In [2]:
from pathlib import Path

project_root = Path.cwd()
if not (project_root / 'app.py').exists():
    project_root = project_root.parent

df = pd.read_csv(project_root / 'data' / 'processed' / 'combined_general.csv')
len(df)


236

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'app.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'program_surveys'
REFERENCE_DIR = PROJECT_ROOT / 'data' / 'reference'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

for p in [RAW_DIR, REFERENCE_DIR, INTERIM_DIR, PROCESSED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

METADATA_COLS = {
    'Номер ответа', 'Дата ответа', 'Время выполнения', 'IP - адрес',
    'Источник распространения', 'Операционная система', 'Браузер', 'Устройство'
}

def strip_qnum(col):
    """Strip leading question number prefix, e.g. '15. Оцените...' → 'Оцените...'"""
    return re.sub(r'^\d+\.\s*', '', col).strip()

def is_teacher_rating_col(col):
    """Individual teacher rating column: 'N. Пожалуйста, оцените работу преподавателей:... - Фамилия Имя'"""
    s = strip_qnum(col)
    return ('оцените работу преподавателей' in s.lower() or
            'оцените работу преподавателей по критериям' in s.lower()) and ' - ' in s

def get_teacher_name(col):
    """Extract teacher name from the end of a teacher rating column header."""
    return strip_qnum(col).rsplit(' - ', 1)[-1].strip()

def is_course_year_col(col):
    return 'выберите курс' in col.lower()

def canonical(col):
    """Map original column header to a normalised column name."""
    if col in METADATA_COLS:
        return col
    if is_course_year_col(col):
        return 'Курс обучения'
    if is_teacher_rating_col(col):
        return f'Оценка преподавателя: {get_teacher_name(col)}'
    return strip_qnum(col)


def load_program(filepath):
    program_name = Path(filepath).stem
    df = pd.read_csv(filepath)

    # Group original column names by their canonical name.
    # Files with year-branching repeat the same question block for each year,
    # so multiple original columns share the same canonical name.
    groups = defaultdict(list)
    for col in df.columns:
        groups[canonical(col)].append(col)

    # Build output series: coalesce duplicates with combine_first.
    out_series = {}
    for cname, orig_cols in groups.items():
        s = df[orig_cols[0]].copy()
        for c in orig_cols[1:]:
            s = s.combine_first(df[c])
        out_series[cname] = s

    out = pd.DataFrame(out_series)

    # Insert program name as first column.
    out.insert(0, 'Программа', program_name)

    # Ensure 'Курс обучения' exists as object dtype (None for single-year programmes).
    if 'Курс обучения' not in out.columns:
        out.insert(1, 'Курс обучения', None)
    else:
        out['Курс обучения'] = out['Курс обучения'].astype(object)

    return out


# ── Load and concatenate all programme CSVs ──────────────────────────────────
dfs = []
for csv_path in sorted(RAW_DIR.glob('*.csv')):
    dfs.append(load_program(csv_path))
    print(f'  loaded: {csv_path.name}')

combined = pd.concat(dfs, ignore_index=True, sort=False)
print(f'\nCombined: {combined.shape[0]} rows × {combined.shape[1]} columns')
combined.head()


  loaded: Foundation Art and Design.csv
  loaded: Анимация.csv
  loaded: Артист драматического театра и кино.csv
  loaded: Архитектура и градостроительство.csv
  loaded: Бренд-менеджмент и маркетинг в креативных индустриях.csv
  loaded: Графический дизайн.csv
  loaded: Дизайн и архитектура интерьера.csv
  loaded: Дизайн одежды и текстиля.csv
  loaded: Иллюстрация.csv
  loaded: Кинопроизводство.csv
  loaded: Компьютерная музыка и аранжировка.csv
  loaded: Менеджмент в креативных индустриях (сетевая программа).csv
  loaded: Предпринимательство и продюсирование в креативных индустриях.csv


/var/folders/xj/n06ls31j69q0md4kzrq5l5km0000gn/T/ipykernel_83793/391932015.py:64: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  s = s.combine_first(df[c])
/var/folders/xj/n06ls31j69q0md4kzrq5l5km0000gn/T/ipykernel_83793/391932015.py:64: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  s = s.combine_first(df[c])
/var/folders/xj/n06ls31j69q0md4kzrq5l5km0000gn/T/ipykernel_83793/391932015.py:64: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the r

  loaded: Проектирование зданий и городских общественных пространств.csv
  loaded: Промышленный дизайн.csv
  loaded: Развитие креативных проектов.csv
  loaded: Современное искусство.csv
  loaded: Создание игр.csv

Combined: 236 rows × 310 columns


/var/folders/xj/n06ls31j69q0md4kzrq5l5km0000gn/T/ipykernel_83793/391932015.py:64: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  s = s.combine_first(df[c])
/var/folders/xj/n06ls31j69q0md4kzrq5l5km0000gn/T/ipykernel_83793/391932015.py:64: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  s = s.combine_first(df[c])
/var/folders/xj/n06ls31j69q0md4kzrq5l5km0000gn/T/ipykernel_83793/391932015.py:64: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the r

,Программа,Курс обучения,Номер ответа,Дата ответа,Время выполнения,IP - адрес,Источник распространения,Операционная система,Браузер,Устройство,...,Оценка преподавателя: Заиграев Сергей,Оценка преподавателя: Ирина Шедько,Оценка преподавателя: Ксения Жернова,Оценка преподавателя: Михаил Лубягов,Оценка преподавателя: Подакин Антон,Оценка преподавателя: Подшибякин Георгий,Оценка преподавателя: Юрий Ситников,Оценка преподавателя: Ксения Плотникова,Оценка преподавателя: Смирнова Анастасия,Оценка преподавателя: Татьяна Полякова
0,Foundation Art and Design,None,1,12.02.2026 14:42:36,73,149.7.104.3,Прямая ссылка,Windows 10,Google Chrome,Компьютер,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Foundation Art and Design,None,2,20.02.2026 20:42:03,270,109.107.166.214,Прямая ссылка,iPhone,Safari,Телефон,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Foundation Art and Design,None,3,20.02.2026 20:44:44,224,5.228.118.254,Прямая ссылка,Android,Google Chrome,Телефон,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Foundation Art and Design,None,4,20.02.2026 20:48:29,630,217.217.244.5,Прямая ссылка,iPhone,Safari,Телефон,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Foundation Art and Design,None,5,20.02.2026 20:50:01,733,5.35.113.119,Прямая ссылка,Android,Google Chrome,Телефон,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ── Reorder columns: metadata first, then common questions, then teacher ratings ─
meta = ['Программа', 'Курс обучения'] + [c for c in METADATA_COLS if c in combined.columns]
teacher_cols = sorted([c for c in combined.columns if c.startswith('Оценка преподавателя:')])
other_cols = [c for c in combined.columns if c not in set(meta) and c not in set(teacher_cols)]

combined = combined[meta + other_cols + teacher_cols]

# ── Save ─────────────────────────────────────────────────────────────────────
out_path = INTERIM_DIR / 'combined.csv'
combined.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print(f'Shape: {combined.shape[0]} rows × {combined.shape[1]} columns')
print(f'  metadata + program info : {len(meta)} cols')
print(f'  survey questions        : {len(other_cols)} cols')
print(f'  individual teacher ratings: {len(teacher_cols)} cols')
print(f'\nProgrammes: {combined["Программа"].nunique()}')
print(combined.groupby("Программа").size().to_string())

Saved → /Users/mtvbsl/Library/CloudStorage/Dropbox/UU/Code/SFQ 2025:26/data/interim/combined.csv
Shape: 236 rows × 310 columns
  metadata + program info : 10 cols
  survey questions        : 62 cols
  individual teacher ratings: 238 cols

Programmes: 18
Программа
Foundation Art and Design                                       31
Анимация                                                         2
Артист драматического театра и кино                              1
Архитектура и градостроительство                                40
Бренд-менеджмент и маркетинг в креативных индустриях            10
Графический дизайн                                             4
Дизайн и архитектура интерьера                                 11
Дизайн одежды и текстиля                                       25
Иллюстрация                                                      8
Кинопроизводство                                                58
Компьютерная музыка и аранжировка                                8

In [5]:
# ── Проверка: сумма строк в индивидуальных файлах == строки в combined ───────
# Используем csv.reader, а не wc -l: открытые вопросы содержат переносы строк
# внутри кавычек — csv.reader считает их одной строкой, wc -l — нет.
import csv as _csv

individual_counts = {}
for csv_path in sorted(RAW_DIR.glob('*.csv')):
    with open(csv_path, encoding='utf-8') as f:
        n = sum(1 for _ in _csv.reader(f)) - 1  # вычитаем заголовок
    individual_counts[csv_path.stem] = n

total_individual = sum(individual_counts.values())
total_combined   = len(combined)

print(f'Строк в индивидуальных файлах (CSV-парсинг): {total_individual}')
print(f'Строк в combined:                            {total_combined}')
print(f'Совпадение: {"✓ ДА" if total_individual == total_combined else "✗ НЕТ — расхождение!"}')
print()
print('Строк по программам:')
for prog, n in individual_counts.items():
    combined_n = len(combined[combined['Программа'] == prog])
    match = '✓' if n == combined_n else f'✗ (combined={combined_n})'
    print(f'  {match}  {n:>3}  {prog}')


Строк в индивидуальных файлах (CSV-парсинг): 236
Строк в combined:                            236
Совпадение: ✓ ДА

Строк по программам:
  ✓   31  Foundation Art and Design
  ✓    2  Анимация
  ✓    1  Артист драматического театра и кино
  ✓   40  Архитектура и градостроительство
  ✓   10  Бренд-менеджмент и маркетинг в креативных индустриях
  ✓    4  Графический дизайн
  ✓   11  Дизайн и архитектура интерьера
  ✓   25  Дизайн одежды и текстиля
  ✓    8  Иллюстрация
  ✓   58  Кинопроизводство
  ✓    8  Компьютерная музыка и аранжировка
  ✓    1  Менеджмент в креативных индустриях (сетевая программа)
  ✓    1  Предпринимательство и продюсирование в креативных индустриях
  ✓   14  Проектирование зданий и городских общественных пространств
  ✓    5  Промышленный дизайн
  ✓    6  Развитие креативных проектов
  ✓   10  Современное искусство
  ✓    1  Создание игр


In [6]:
# ── Разбивка на два файла ─────────────────────────────────────────────────────
teacher_cols = [c for c in combined.columns if c.startswith('Оценка преподавателя:')]
id_cols      = ['Программа', 'Курс обучения', 'Номер ответа']

# 1. Общие вопросы (без оценок индивидуальных преподавателей)
general_cols = [c for c in combined.columns if c not in teacher_cols]
df_general = combined[general_cols]
df_general.to_csv(PROCESSED_DIR / 'combined_general.csv', index=False)
print(f'combined_general.csv — {df_general.shape[0]} строк × {df_general.shape[1]} колонок')

# 2. Оценки преподавателей в длинном формате (long-form):
#    каждая строка = один ответ × один преподаватель
df_teachers = (
    combined[id_cols + teacher_cols]
    .melt(id_vars=id_cols, var_name='Преподаватель', value_name='Оценка')
    .dropna(subset=['Оценка'])
)
df_teachers['Преподаватель'] = df_teachers['Преподаватель'].str.removeprefix('Оценка преподавателя: ')
df_teachers.to_csv(PROCESSED_DIR / 'combined_teachers.csv', index=False)
print(f'combined_teachers.csv  — {df_teachers.shape[0]} строк × {df_teachers.shape[1]} колонок')
print(f'  Уникальных преподавателей: {df_teachers["Преподаватель"].nunique()}')
df_teachers.head(8)

combined_general.csv — 236 строк × 72 колонок
combined_teachers.csv  — 2321 строк × 5 колонок
  Уникальных преподавателей: 204


,Программа,Курс обучения,Номер ответа,Преподаватель,Оценка
31,Анимация,1 курс,1,Dan Smith,10.0
32,Анимация,1 курс,2,Dan Smith,10.0
33,Артист драматического театра и кино,None,1,Dan Smith,1.0
132,Кинопроизводство,1 курс,1,Dan Smith,9.0
133,Кинопроизводство,1 курс,2,Dan Smith,8.0
135,Кинопроизводство,1 курс,4,Dan Smith,1.0
136,Кинопроизводство,1 курс,5,Dan Smith,4.0
143,Кинопроизводство,1 курс,12,Dan Smith,8.0


In [7]:
# ── Rename columns to short Latin names ──────────────────────────────────────
# Format: {original_canonical_name: short_name}
COL_MAP = {
    # ── identifiers & metadata ──────────────────────────────────────────────
    'Программа':                                            'program',
    'Курс обучения':                                        'year',
    'Номер ответа':                                         'resp_id',
    'Дата ответа':                                          'date',
    'Время выполнения':                                     'duration_sec',
    'IP - адрес':                                           'ip',
    'Источник распространения':                             'source',
    'Операционная система':                                 'os',
    'Браузер':                                              'browser',
    'Устройство':                                           'device',

    # ── Q1–3: global satisfaction ───────────────────────────────────────────
    'Пожалуйста, оцените вашу удовлетворенность образовательным процессом в целом':
        'satisf_overall',
    'Насколько вы в целом удовлетворены работой преподавателей в течение семестра?':
        'satisf_teachers',
    'Насколько ваши ожидания от программы при поступлении совпали с опытом обучения?':
        'expect_match',

    # ── Q4: faculty team (matrix) ───────────────────────────────────────────
    'Пожалуйста, оцените вашу удовлетворенность преподавательским составом на программе - У меня была достаточная поддержка преподавательской команды в профессиональном развитии - Оценка':
        'fac_support_score',
    'Пожалуйста, оцените вашу удовлетворенность преподавательским составом на программе - У меня была достаточная поддержка преподавательской команды в профессиональном развитии - Важность критерия':
        'fac_support_imp',
    'Пожалуйста, оцените вашу удовлетворенность преподавательским составом на программе - Преподавательская команда объясняет сложные вещи понятным языком - Оценка':
        'fac_clarity_score',
    'Пожалуйста, оцените вашу удовлетворенность преподавательским составом на программе - Преподавательская команда объясняет сложные вещи понятным языком - Важность критерия':
        'fac_clarity_imp',

    # ── Q5: curator (matrix) ────────────────────────────────────────────────
    'Пожалуйста, оцените вашу удовлетворенность куратором на программе - Куратор своевременно сообщает необходимую информацию об образовательном процессе - Оценка':
        'cur_timely_score',
    'Пожалуйста, оцените вашу удовлетворенность куратором на программе - Куратор своевременно сообщает необходимую информацию об образовательном процессе - Важность критерия':
        'cur_timely_imp',
    'Пожалуйста, оцените вашу удовлетворенность куратором на программе - Куратор готов оказать помощь, если у меня появляются вопросы или проблемы - Оценка':
        'cur_help_score',
    'Пожалуйста, оцените вашу удовлетворенность куратором на программе - Куратор готов оказать помощь, если у меня появляются вопросы или проблемы - Важность критерия':
        'cur_help_imp',
    'Здесь вы можете оставить более подробную обратную связь о кураторе':
        'cur_comment',

    # ── Q7: programme content (matrix) ─────────────────────────────────────
    'Пожалуйста, оцените вашу удовлетворенность программой по каждому из параметров - Мне было понятно содержание дисциплин, которые я изучал(а) в течение семестра, и их взаимосвязь между собой - Оценка':
        'prog_clarity_score',
    'Пожалуйста, оцените вашу удовлетворенность программой по каждому из параметров - Мне было понятно содержание дисциплин, которые я изучал(а) в течение семестра, и их взаимосвязь между собой - Важность критерия':
        'prog_clarity_imp',
    'Пожалуйста, оцените вашу удовлетворенность программой по каждому из параметров - Я мог(-ла) выполнить задания преподавателей в нужные сроки в достаточном объеме - Оценка':
        'prog_deadlines_score',
    'Пожалуйста, оцените вашу удовлетворенность программой по каждому из параметров - Я мог(-ла) выполнить задания преподавателей в нужные сроки в достаточном объеме - Важность критерия':
        'prog_deadlines_imp',
    'Пожалуйста, оцените вашу удовлетворенность программой по каждому из параметров - Мне было понятно, как изучаемые дисциплины связаны с моей сферой профессиональной деятельности - Оценка':
        'prog_relevance_score',
    'Пожалуйста, оцените вашу удовлетворенность программой по каждому из параметров - Мне было понятно, как изучаемые дисциплины связаны с моей сферой профессиональной деятельности - Важность критерия':
        'prog_relevance_imp',
    'Пожалуйста, оцените вашу удовлетворенность программой по каждому из параметров - Мне кажется, что за прошедший семестр количество занятий в неделю было оптимальным - Оценка':
        'prog_workload_score',
    'Пожалуйста, оцените вашу удовлетворенность программой по каждому из параметров - Мне кажется, что за прошедший семестр количество занятий в неделю было оптимальным - Важность критерия':
        'prog_workload_imp',
    'Здесь вы можете оставить более подробную обратную связь о программе':
        'prog_comment',

    # ── Q9: study office coordinator (matrix) ───────────────────────────────
    'Пожалуйста, оцените вашу удовлетворенность взаимодействием с координатором Учебного отдела программы - Координатор уважительно взаимодействовал со мной - Оценка':
        'coord_respect_score',
    'Пожалуйста, оцените вашу удовлетворенность взаимодействием с координатором Учебного отдела программы - Координатор уважительно взаимодействовал со мной - Важность критерия':
        'coord_respect_imp',
    'Пожалуйста, оцените вашу удовлетворенность взаимодействием с координатором Учебного отдела программы - Мои обращения к координатору приводили к ожидаемому результату - Оценка':
        'coord_results_score',
    'Пожалуйста, оцените вашу удовлетворенность взаимодействием с координатором Учебного отдела программы - Мои обращения к координатору приводили к ожидаемому результату - Важность критерия':
        'coord_results_imp',
    'Пожалуйста, оцените вашу удовлетворенность взаимодействием с координатором Учебного отдела программы - Координатор сообщает необходимую информацию об образовательном процессе своевременно - Оценка':
        'coord_timely_score',
    'Пожалуйста, оцените вашу удовлетворенность взаимодействием с координатором Учебного отдела программы - Координатор сообщает необходимую информацию об образовательном процессе своевременно - Важность критерия':
        'coord_timely_imp',
    'Пожалуйста, оцените вашу удовлетворенность взаимодействием с координатором Учебного отдела программы - Координатор готов оказать помощь, если возникают вопросы или проблемы - Оценка':
        'coord_help_score',
    'Пожалуйста, оцените вашу удовлетворенность взаимодействием с координатором Учебного отдела программы - Координатор готов оказать помощь, если возникают вопросы или проблемы - Важность критерия':
        'coord_help_imp',
    'Здесь вы можете оставить более подробную обратную связь о взаимодействии с координатором Учебного отдела':
        'coord_comment',

    # ── Q11–13: assessment ──────────────────────────────────────────────────
    'Преподаватели своевременно представили критерии оценивания работ':
        'assess_criteria_timely',
    'Преподаватели ясно определили порядок сдачи и оценивания работ':
        'assess_order_clear',
    'Преподаватели проводили оценивание работ в соответствии с заявленными критериями':
        'assess_consistent',
    'Здесь вы можете оставить подробную обратную связь про оценивание на программе':
        'assess_comment',
    'Здесь вы можете оставить более подробную обратную связь о преподавательской команде':
        'fac_comment',

    # ── Q17/18: general education & humanities teachers ─────────────────────
    # "общеобразовательные" (Foundation Art and Design only)
    'Оцените преподавателей по общеобразовательным дисциплинам - Критическое мышление': 'gen_ed_critical_thinking',
    'Оцените преподавателей по общеобразовательным дисциплинам - История России':        'gen_ed_history',
    'Оцените преподавателей по общеобразовательным дисциплинам - Иностранный язык':      'gen_ed_foreign_lang',
    'Оцените преподавателей по общеобразовательным дисциплинам - Безопасность жизнедеятельности': 'gen_ed_safety',
    'Оцените преподавателей по общеобразовательным дисциплинам - Основы российской государственности': 'gen_ed_statehood',
    # "гуманитарные" (all other programmes)
    'Оцените преподавателей по гуманитарным дисциплинам: - Критическое мышление':        'hum_critical_thinking',
    'Оцените преподавателей по гуманитарным дисциплинам: - История России':               'hum_history',
    'Оцените преподавателей по гуманитарным дисциплинам: - Основы российской государственности': 'hum_statehood',
    'Оцените преподавателей по гуманитарным дисциплинам: - Иностранный язык':             'hum_foreign_lang',
    'Оцените преподавателей по гуманитарным дисциплинам: - Философия':                    'hum_philosophy',
    'Оцените преподавателей по гуманитарным дисциплинам: - Теория и практика коммуникации': 'hum_communication',

    # ── seminar comment ─────────────────────────────────────────────────────
    'Оставьте комментарий отдельно о лекционных и семинарских занятиях (укажите преподавателя или семинарский трек). Насколько разнообразными и полезными были семинарские треки:':
        'seminar_comment',

    # ── Q19–25: campus infrastructure ───────────────────────────────────────
    'Оцените доступность и работу библиотеки':                                  'infra_library',
    'Оцените доступность и качество работы сервиса Wellbeing':                  'infra_wellbeing',
    'Оцените доступность и качество еды на кампусе':                            'infra_food',
    'Оцените работоспособность программного обеспечения в аудиториях':          'infra_software',
    'Оцените работоспособность оборудования в аудиториях':                      'infra_equipment',
    'Оцените комфорт пребывания в аудиториях':                                  'infra_classrooms',
    'Оцените комфорт пребывания в мастерских / ресурсных центрах / репетиционных комнатах':
        'infra_workshops',

    # ── Q26: NPS & final comment ────────────────────────────────────────────
    'Насколько вероятно, что вы порекомендуете школу друзьям, коллегам или родным, которые планируют обучение':
        'nps',
    'Есть ли у вас дополнительные комментарии или предложения для нас?':
        'comment_final',

    # ── additional questions (subset of programmes) ─────────────────────────
    'Насколько дисциплины предыдущего семестра помогли вам в освоении учебного материала текущего семестра?':
        'prev_sem_relevance',
    'Насколько уверенно вы можете применить полученные знания и навыки в реальной профессиональной практике?':
        'skill_confidence',
    'Как вы планируете продолжать свое профессиональное развитие после выпуска? - Продолжение обучения в магистратуре или дополнительном образовании':
        'postgrad_masters',
    'Как вы планируете продолжать свое профессиональное развитие после выпуска? - Работа в выбранной сфере':
        'postgrad_same_field',
    'Как вы планируете продолжать свое профессиональное развитие после выпуска? - Работа в другой сфере':
        'postgrad_other_field',
    'Как вы планируете продолжать свое профессиональное развитие после выпуска? -':
        'postgrad_other',
}

# Verify all columns are mapped
unmapped = [c for c in df_general.columns if c not in COL_MAP]
if unmapped:
    print('⚠ Unmapped columns:')
    for c in unmapped:
        print(f'  {repr(c)}')
else:
    print('✓ All columns mapped')

df_agg = df_general.rename(columns=COL_MAP)

# ── Enrich from programme registry: school + missing year ──────────────────
import unicodedata

def normalize_text(s):
    if pd.isna(s):
        return np.nan
    s = unicodedata.normalize('NFKC', str(s)).strip().lower().replace('ё', 'е')
    return ' '.join(s.split())

registry_path = REFERENCE_DIR / 'Реестр программ.csv'
if os.path.exists(registry_path):
    registry = pd.read_csv(registry_path, sep=';')
    needed_cols = {'Программа', 'Школа', 'Курс'}

    if needed_cols.issubset(registry.columns):
        registry = registry.copy()
        registry['program_norm'] = registry['Программа'].map(normalize_text)
        registry['Курс'] = pd.to_numeric(registry['Курс'], errors='coerce')

        df_agg['program_norm'] = df_agg['program'].map(normalize_text)

        # school: unique mapping per programme
        school_map = (
            registry[['program_norm', 'Школа']]
            .dropna(subset=['program_norm'])
            .drop_duplicates()
            .groupby('program_norm')['Школа']
            .agg(lambda s: s.iloc[0] if s.nunique() == 1 else np.nan)
        )
        df_agg['school'] = df_agg['program_norm'].map(school_map)

        # year fill: only when registry has a single course for programme
        course_candidates = (
            registry[['program_norm', 'Курс']]
            .dropna(subset=['program_norm', 'Курс'])
            .drop_duplicates()
            .groupby('program_norm')['Курс']
            .apply(lambda s: sorted(set(int(v) for v in s if pd.notna(v))))
        )
        single_course_map = {k: v[0] for k, v in course_candidates.items() if len(v) == 1}

        missing_year_mask = df_agg['year'].isna() | (df_agg['year'].astype(str).str.strip() == '')
        inferred_year = (
            df_agg.loc[missing_year_mask, 'program_norm']
            .map(single_course_map)
            .map(lambda x: f'{int(x)} курс' if pd.notna(x) else np.nan)
        )
        df_agg.loc[missing_year_mask, 'year'] = inferred_year

        unresolved_school = int(df_agg['school'].isna().sum())
        unresolved_year = int((df_agg['year'].isna() | (df_agg['year'].astype(str).str.strip() == '')).sum())
        df_agg = df_agg.drop(columns=['program_norm'])

        print(f'✓ Registry applied from: {registry_path}')
        print(f'  Missing school after merge: {unresolved_school}')
        print(f'  Missing year after fill: {unresolved_year}')
    else:
        print('⚠ Registry file found, but required columns are missing:', needed_cols - set(registry.columns))
        df_agg['school'] = np.nan
else:
    print(f'⚠ Registry file not found: {registry_path}')
    df_agg['school'] = np.nan

df_agg.to_csv(PROJECT_ROOT / 'combined_general_agg.csv', index=False)
print(f'Saved combined_general_agg.csv  ({df_agg.shape[0]} rows × {df_agg.shape[1]} cols)')
df_agg.head(3)


✓ All columns mapped
✓ Registry applied from: /Users/mtvbsl/Library/CloudStorage/Dropbox/UU/Code/SFQ 2025:26/data/reference/Реестр программ.csv
  Missing school after merge: 0
  Missing year after fill: 0
Saved combined_general_agg.csv  (236 rows × 73 cols)


,program,year,duration_sec,device,browser,os,ip,date,resp_id,source,...,hum_foreign_lang,prev_sem_relevance,postgrad_masters,postgrad_same_field,postgrad_other_field,postgrad_other,hum_philosophy,hum_communication,skill_confidence,school
0,Foundation Art and Design,1 курс,73,Компьютер,Google Chrome,Windows 10,149.7.104.3,12.02.2026 14:42:36,1,Прямая ссылка,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,БВШД
1,Foundation Art and Design,1 курс,270,Телефон,Safari,iPhone,109.107.166.214,20.02.2026 20:42:03,2,Прямая ссылка,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,БВШД
2,Foundation Art and Design,1 курс,224,Телефон,Google Chrome,Android,5.228.118.254,20.02.2026 20:44:44,3,Прямая ссылка,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,БВШД
